#### Import Libraries

In [2]:
import argparse
import pandas as pd
import io
import argparse
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

#### Regional Summary

In [3]:
# regional_summary.py
# Aggregate municipal EV charger estimates (rf/xgb_EV_area_distr.csv) into the
# five Danish regions (NUTS 2, cf. https://www.dst.dk/en/Statistik/dokumentation/nomenklaturer/nuts)
# with a selectable probability threshold.
#
# Usage (as a script):
#   python regional_summary.py                      # both models, tau = 0.90
#   python regional_summary.py --threshold 70       # tau = 0.70
#   python regional_summary.py --model rf --threshold 95
#
# Output: printed table + <model>_regional_p<threshold>.csv (for the map later).

# --- Municipality (KOM) -> Region, per DST NUTS nomenclature ---------------
# Verify against the DST source before final use.
REGIONS = {
    "Hovedstaden": [
        101, 147, 151, 153, 155, 157, 159, 161, 163, 165, 167, 169, 173, 175,
        183, 185, 187, 190, 201, 210, 217, 219, 223, 230, 240, 250, 260, 270,
        400,  # Bornholm
    ],
    "Sjælland": [
        253, 259, 265, 269, 306, 316, 320, 326, 329, 330, 336, 340, 350, 360,
        370, 376, 390,
    ],
    "Syddanmark": [
        410, 420, 430, 440, 450, 461, 479, 480, 482, 492, 510, 530, 540, 550,
        561, 563, 573, 575, 580, 607, 621, 630,
    ],
    "Midtjylland": [
        615, 657, 661, 665, 671, 706, 707, 710, 727, 730, 740, 741, 746, 751,
        756, 760, 766, 779, 791,
    ],
    "Nordjylland": [
        773, 787, 810, 813, 820, 825, 840, 846, 849, 851, 860,
    ],
}
KOM_TO_REGION = {kom: reg for reg, koms in REGIONS.items() for kom in koms}

VALID_THRESHOLDS = (50, 70, 80, 90, 95)


def regional_summary(csv_path: str, threshold: int = 90) -> pd.DataFrame:
    """Sum municipal estimates into regions for the selected threshold (50/70/80/90/95)."""
    if threshold not in VALID_THRESHOLDS:
        raise ValueError(f"threshold must be one of {VALID_THRESHOLDS}")
    col = f"n_evs_p{threshold}"

    df = pd.read_csv(csv_path)
    df["region"] = df["KOM"].map(KOM_TO_REGION)

    unmapped = df.loc[df["region"].isna(), "KOM"].tolist()
    if unmapped:
        raise ValueError(f"KOM codes without region mapping: {unmapped}")

    out = (
        df.groupby("region")
          .agg(n_municipalities=("KOM", "size"),
               n_meters=("n_meters", "sum"),
               n_evs=(col, "sum"))
          .reindex(REGIONS.keys())
    )
    out["penetration_pct"] = (out["n_evs"] / out["n_meters"] * 100).round(2)
    # meter-weighted mean of the annual mean probability, for reference
    w = df["n_meters"] * df["mean_yearly_ev_probability"]
    out["weighted_mean_prob"] = (
        w.groupby(df["region"]).sum() / df.groupby("region")["n_meters"].sum()
    ).reindex(REGIONS.keys()).round(4)
    # country-level total: meter-weighted national penetration, for reference
    out.loc["Denmark"] = [
        out["n_municipalities"].sum(), out["n_meters"].sum(), out["n_evs"].sum(),
        round(out["n_evs"].sum() / out["n_meters"].sum() * 100, 2),
        round(w.sum() / df["n_meters"].sum(), 4),
    ]
    out[["n_municipalities", "n_meters", "n_evs"]] = (
        out[["n_municipalities", "n_meters", "n_evs"]].astype(int)
    )
    return out


threshold = 90
for m in ["rf", "xgb"]:
    res = regional_summary(f"../data/{m}_EV_area_distr.csv", threshold)
    print(f"\n=== {m.upper()} — regional estimates at P >= {threshold}% ===")
    print(res.to_string())
    out_csv = f"../data/{m}_regional_p{threshold}.csv"
    res.to_csv(out_csv)
    print(f"saved {out_csv}")



=== RF — regional estimates at P >= 90% ===
             n_municipalities  n_meters   n_evs  penetration_pct  weighted_mean_prob
region                                                                              
Hovedstaden                29    763217   23103             3.03              0.2300
Sjælland                   17    340701   16201             4.76              0.3574
Syddanmark                 22    502244   22858             4.55              0.3531
Midtjylland                19    536192   29531             5.51              0.3571
Nordjylland                11    252436   11102             4.40              0.3425
Denmark                    98   2394790  102795             4.29              0.3142
saved ../data/rf_regional_p90.csv

=== XGB — regional estimates at P >= 90% ===
             n_municipalities  n_meters   n_evs  penetration_pct  weighted_mean_prob
region                                                                              
Hovedstaden              

#### Regional Map

In [4]:
# make_regional_map.py
# Choropleth ("heatmap") of estimated EV charger penetration for the
# five Danish regions, at a selectable probability threshold.
# One figure per model (shared colour scale across models for comparability).
# Boundaries: official DAWA API (public). Uses regional_summary.py (same folder).
#
# Usage:
#   python make_regional_map.py                          # both models, tau=0.90
#   python make_regional_map.py --model xgb --threshold 70
#
# Requirements: pip install geopandas matplotlib requests

import matplotlib.patheffects as pe

from regional_summary import regional_summary, VALID_THRESHOLDS

GEO_URL = "https://api.dataforsyningen.dk/regioner?format=geojson"

# DAWA region names -> names used in regional_summary.REGIONS
DAWA_TO_REGION = {
    "Nordjylland": "Nordjylland",
    "Midtjylland": "Midtjylland",
    "Syddanmark": "Syddanmark",
    "Hovedstaden": "Hovedstaden",
    "Sjælland": "Sjælland",
}


def load_boundaries() -> gpd.GeoDataFrame:
    r = requests.get(GEO_URL, timeout=120)
    r.raise_for_status()
    gdf = gpd.read_file(io.BytesIO(r.content))
    gdf["region"] = gdf["navn"].str.replace("Region ", "", regex=False).map(DAWA_TO_REGION)
    gdf["geometry"] = gdf.geometry.simplify(0.005)  # lighter rendering
    return gdf


def plot(models, threshold, cmap="PuBuGn"):
    gdf = load_boundaries()

    # shared colour scale across models so the separate maps stay comparable
    tables = {m: regional_summary(f"../data/{m}_EV_area_distr.csv", threshold) for m in models}
    vmax = max(t["penetration_pct"].max() for t in tables.values())

    for m in models:
        fig, ax = plt.subplots(figsize=(8, 8))
        t = tables[m]
        g = gdf.merge(t, left_on="region", right_index=True)
        g.plot(column="penetration_pct", ax=ax, cmap=cmap, vmin=0, vmax=vmax,
               edgecolor="white", linewidth=0.8,
               legend=True,
               legend_kwds={"label": f"EV charger penetration (% of meters), "
                                     f"$\\tau={threshold/100:.2f}$", "shrink": 0.6})
        # annotate each region with name and value; bold white with a thin black
        # outline stays readable on any map shade and on the white background
        for _, row in g.iterrows():
            c = row.geometry.representative_point()
            ax.annotate(f"{row['region']}\n{row['penetration_pct']:.1f}%",
                        xy=(c.x, c.y), ha="center", fontsize=9,
                        color="white", fontweight="bold",
                        path_effects=[pe.withStroke(linewidth=1.5, foreground="black")])
        ax.set_title({"rf": "EV Chargers Distribution per Region - Random Forest",
                      "xgb": "EV Chargers Distribution per Region - XGBoost"}[m])
        ax.set_axis_off()

        plt.tight_layout()
        out_pdf = f"../results/regional_map_{m}_p{threshold}.pdf"
        plt.savefig(out_pdf, bbox_inches="tight", dpi=300)
        plt.close(fig)
        print(f"saved {out_pdf}")


plot(["rf", "xgb"], 90)


saved ../results/regional_map_rf_p90.pdf
saved ../results/regional_map_xgb_p90.pdf


#### Municipality Map

In [5]:
# make_municipality_map.py
# Choropleth of estimated EV charger penetration per municipality (tau = 0.90).
# One figure per model (shared colour scale across models for comparability).
# Requirements: pip install geopandas matplotlib requests
# Boundaries: official DAWA / Dataforsyningen API (public), joined on KOM code.



RF_CSV  = "../data/rf_EV_area_distr.csv"
XGB_CSV = "../data/xgb_EV_area_distr.csv"
GEO_URL = "https://api.dataforsyningen.dk/kommuner?format=geojson"

# --- load model outputs -------------------------------------------------
def load(path, label):
    df = pd.read_csv(path)
    df["pen90"] = df["n_evs_p90"] / df["n_meters"] * 100.0
    return df[["KOM", "pen90"]].rename(columns={"pen90": label})

data = load(RF_CSV, "RF").merge(load(XGB_CSV, "XGB"), on="KOM")

# --- load municipal boundaries ------------------------------------------
# DAWA 'kode' is a zero-padded string ("0101"); convert to int to match KOM.
r = requests.get(GEO_URL, timeout=120)
r.raise_for_status()
gdf = gpd.read_file(io.BytesIO(r.content))
gdf["KOM"] = gdf["kode"].astype(int)
gdf = gdf.merge(data, on="KOM", how="left")
# lighter file / faster rendering (tolerance in degrees; boundaries stay visually intact)
gdf["geometry"] = gdf.geometry.simplify(0.002)

# --- plot: one figure per model, shared colour scale ---------------------
vmax = gdf[["RF", "XGB"]].max().max()
for col, title, suffix in [("RF", "EV Chargers Distribution per Municipality - Random Forest", "rf"), ("XGB", "EV Chargers Distribution per Municipality - XGBoost", "xgb")]:
    fig, ax = plt.subplots(figsize=(8, 8))
    gdf.plot(column=col, ax=ax, cmap="viridis", vmin=0, vmax=vmax,
             edgecolor="white", linewidth=0.3,
             legend=True,
             legend_kwds={"label": "Estimated EV charger penetration (% of meters), $\\tau=0.90$",
                          "shrink": 0.6},
             missing_kwds={"color": "lightgrey"})
    ax.set_title(title)
    ax.set_axis_off()

    plt.tight_layout()
    out_pdf = f"../results/municipality_choropleth_{suffix}.pdf"
    plt.savefig(out_pdf, bbox_inches="tight", dpi=300)
    plt.close(fig)
    print(f"saved {out_pdf}")


saved ../results/municipality_choropleth_rf.pdf
saved ../results/municipality_choropleth_xgb.pdf
